In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# PATH
# ============================================================

base_path = Path(r"F:\# study 2\eeg_data\epochs\source")

# ============================================================
# COLLECT EPOCH COUNTS
# ============================================================

rows = []

for group_dir in base_path.iterdir():

    if not group_dir.is_dir():
        continue

    group = group_dir.name

    # Subject folders, e.g. s1_pac_sub12
    for sub_dir in group_dir.iterdir():

        if not sub_dir.is_dir():
            continue

        sub = sub_dir.name

        # Task folders, e.g. FTT / DeCRAT
        for task_dir in sub_dir.iterdir():

            if not task_dir.is_dir():
                continue

            task = task_dir.name

            # Stage folders, e.g. plan / go
            for stage_dir in task_dir.iterdir():

                if not stage_dir.is_dir():
                    continue

                task_stage = stage_dir.name

                # ------------------------------------------------
                # FTT: no condition subfolder
                # ------------------------------------------------
                if task == "FTT":

                    n_stc = len(list(stage_dir.glob("*.stc")))
                    n_epochs = n_stc / 2

                    rows.append({
                        "sub": sub,
                        "group": group,
                        "task": task,
                        "task_stage": task_stage,
                        "condition": np.nan,
                        "n_epochs": n_epochs
                    })

                # ------------------------------------------------
                # Other tasks: condition subfolders
                # ------------------------------------------------
                else:

                    for condition_dir in stage_dir.iterdir():

                        if not condition_dir.is_dir():
                            continue

                        condition = condition_dir.name

                        n_stc = len(list(condition_dir.glob("*.stc")))
                        n_epochs = n_stc / 2

                        rows.append({
                            "sub": sub,
                            "group": group,
                            "task": task,
                            "task_stage": task_stage,
                            "condition": condition,
                            "n_epochs": n_epochs
                        })


# ============================================================
# MASTER DATAFRAME
# ============================================================

stc = pd.DataFrame(rows)

stc = stc.sort_values(
    ["group", "sub", "task", "task_stage", "condition"],
    na_position="first"
).reset_index(drop=True)


# ============================================================
# CHECK RESULTS
# ============================================================

print("\n============================================================")
print("STC EPOCH COUNTS")
print("============================================================\n")

print(stc)

print("\nNumber of observations:", len(stc))
print("Number of subjects:", stc["sub"].nunique())


# ============================================================
# DESCRIPTIVE STATISTICS BY AGE GROUP
# ============================================================

print("\n============================================================")
print("EPOCH COUNTS BY AGE GROUP")
print("============================================================\n")

group_descriptives = (
    stc
    .groupby("group")["n_epochs"]
    .describe()
)

print(group_descriptives)


# ============================================================
# COMPACT GROUP SUMMARY
# ============================================================

group_summary = (
    stc
    .groupby("group")["n_epochs"]
    .agg(
        N="count",
        Mean="mean",
        SD="std",
        Median="median",
        Min="min",
        Max="max"
    )
    .reset_index()
)

print("\n============================================================")
print("GROUP SUMMARY")
print("============================================================\n")

print(group_summary)


# ============================================================
# SAVE MASTER DATAFRAME
# ============================================================

output_file = base_path / "stc_epoch_counts.csv"

stc.to_csv(
    output_file,
    index=False
)

print("\nSaved:")
print(output_file)


STC EPOCH COUNTS

              sub group    task task_stage   condition  n_epochs
0    s1_pac_sub12     O  DeCRAT         go  adaptation     187.0
1    s1_pac_sub12     O  DeCRAT         go    baseline      62.0
2    s1_pac_sub12     O  DeCRAT       plan  adaptation     135.0
3    s1_pac_sub12     O  DeCRAT       plan    baseline      45.0
4    s1_pac_sub12     O     FTT         go         NaN      97.0
..            ...   ...     ...        ...         ...       ...
277  s1_pac_sub77     Y  DeCRAT         go    baseline      58.0
278  s1_pac_sub77     Y  DeCRAT       plan  adaptation     127.0
279  s1_pac_sub77     Y  DeCRAT       plan    baseline      42.0
280  s1_pac_sub77     Y     FTT         go         NaN     110.0
281  s1_pac_sub77     Y     FTT       plan         NaN     114.0

[282 rows x 6 columns]

Number of observations: 282
Number of subjects: 47

EPOCH COUNTS BY AGE GROUP

       count        mean        std   min   25%    50%     75%    max
group                      

In [3]:
print(group_summary)

  group    N        Mean         SD  Median   Min    Max
0     O  144  102.673611  44.992436    99.0  40.0  190.0
1     Y  138  105.688406  46.598424   103.5  42.0  191.0


In [4]:
stc

,sub,group,task,task_stage,condition,n_epochs
0,s1_pac_sub12,O,DeCRAT,go,adaptation,187.0
1,s1_pac_sub12,O,DeCRAT,go,baseline,62.0
2,s1_pac_sub12,O,DeCRAT,plan,adaptation,135.0
3,s1_pac_sub12,O,DeCRAT,plan,baseline,45.0
4,s1_pac_sub12,O,FTT,go,NaN,97.0
...,...,...,...,...,...,...
277,s1_pac_sub77,Y,DeCRAT,go,baseline,58.0
278,s1_pac_sub77,Y,DeCRAT,plan,adaptation,127.0
279,s1_pac_sub77,Y,DeCRAT,plan,baseline,42.0
280,s1_pac_sub77,Y,FTT,go,NaN,110.0


In [12]:
stc.groupby(["group", "task", "task_stage", "condition"], dropna=False)["n_epochs"].describe()

count        mean        std    min  \
group task   task_stage condition                                         
O     DeCRAT go         adaptation   24.0  176.833333  16.064183  115.0   
                        baseline     24.0   59.916667   3.798360   49.0   
             plan       adaptation   24.0  135.625000   8.318719  113.0   
                        baseline     24.0   47.250000   2.893020   40.0   
      FTT    go         NaN          24.0   95.125000   9.611191   64.0   
             plan       NaN          24.0  101.291667   9.866408   72.0   
Y     DeCRAT go         adaptation   23.0  184.260870   5.495597  168.0   
                        baseline     23.0   60.782609   3.218646   50.0   
             plan       adaptation   23.0  137.217391   4.833186  127.0   
                        baseline     23.0   47.347826   2.080400   42.0   
      FTT    go         NaN          23.0   98.260870  10.086974   72.0   
             plan       NaN          23.0  106.260870  11.572763   74.0   

                                       25%    50%     75%    max  
group task   task_stage condition                                 
O     DeCRAT go         adaptation  176.00  182.0  185.25  190.0  
                        baseline     58.75   61.0   62.25   64.0  
             plan       adaptation  132.75  138.0  141.25  145.0  
                        baseline     46.00   46.5   49.25   52.0  
      FTT    go         NaN          94.50   96.5  100.00  107.0  
             plan       NaN          98.75  103.0  107.25  113.0  
Y     DeCRAT go         adaptation  182.50  186.0  188.00  191.0  
                        baseline     59.50   62.0   63.00   64.0  
             plan       adaptation  134.00  138.0  141.00  143.0  
                        baseline     46.00   48.0   49.00   50.0  
      FTT    go         NaN          96.50   99.0  106.00  110.0  
             plan       NaN         100.50  108.0  113.50  119.0